# 🌐 MDAAR Free BYOK Multi-Provider Router (2nd Colab / Self-Hosted Hub)
### راوتر مفاتيح مجاني بلا حدود للجمع بين Gemini وGroq وOpenRouter وMistral وCerebras في رابط واحد
---
**لماذا هذا الدفتر؟**
- يتيح لك تشغيل خادم توجيه خفيف متوافق مع OpenAI (`/v1/chat/completions`) على **حساب Google Colab آخر** (بدون حاجة لـ GPU أو مكتبات ثقيلة!) وربطه عبر نفق Cloudflare الآمن بدفتر **MDAAR** الرئيسي أو بأي أداة برمجية.
- ضع جميع مفاتيحك المجانية في **Colab Secrets (🔑)** داخل هذا الدفتر فقط، وسيتم تدوير المفاتيح تلقائياً والانتقال الفوري بين المزودين والموديلات عند أي خطأ `429` أو `404`، **بلا أي حدود وسيطة من طرف ثالث**.

### 🔑 جدول أسرار كولاب (`Colab Secrets`) المدعومة
| اسم المتغير / Secret | الوظيفة والافتراضي |
|:---|:---|
| `MDAAR_ROUTER_TOKEN` | توكن الحماية Bearer Token لربط النفق بدفتر MDAAR بأمان |
| `MDAAR_ROUTER_PORT` | منفذ التشغيل المحلي (الافتراضي `8787`) |
| `MDAAR_ROUTER_COOLDOWN` | مدة التهدئة بالثواني عند `429` (الافتراضي `45`) |
| `MDAAR_ROUTER_GEMINI_KEYS` أو `GEMINI_API_KEY` | مفاتيح Google AI Studio (مفصولة بفاصلة `,`) |
| `MDAAR_ROUTER_GEMINI_MODELS` | مودلات Gemini/Gemma (افتراضياً كل السلسلة المجانية `auto-free-chain`) |
| `MDAAR_ROUTER_GROQ_KEYS` أو `GROQ_API_KEY` | مفاتيح Groq المجانية (مفصولة بفاصلة) |
| `MDAAR_ROUTER_GROQ_MODELS` | الافتراضي `llama-3.3-70b-versatile,llama-3.1-8b-instant` |
| `MDAAR_ROUTER_OPENROUTER_KEYS` أو `OPENROUTER_API_KEY` | مفاتيح OpenRouter المجانية |
| `MDAAR_ROUTER_OPENROUTER_MODELS` | الافتراضي `openrouter/free` |
| `MDAAR_ROUTER_MISTRAL_KEYS` أو `MISTRAL_API_KEY` | مفاتيح Mistral AI |
| `MDAAR_ROUTER_MISTRAL_MODELS` | الافتراضي `mistral-small-latest` |
| `MDAAR_ROUTER_CEREBRAS_KEYS` أو `CEREBRAS_API_KEY` | مفاتيح Cerebras المجانية |
| `MDAAR_ROUTER_CEREBRAS_MODELS` | الافتراضي `llama-3.3-70b` |
| `MDAAR_ROUTER_CUSTOM_NAME` | اسم المسار المخصص (الافتراضي `custom`) |
| `MDAAR_ROUTER_CUSTOM_BASE_URL` | رابط أي مزود/راوتر إضافي متوافق مع OpenAI |
| `MDAAR_ROUTER_CUSTOM_KEYS` | مفاتيح المزود المخصص |
| `MDAAR_ROUTER_CUSTOM_MODELS` | أسماء مودلات المزود المخصص |

In [ ]:
#@title 1. تثبيت MDAAR (وضع خفيف بدون GPU) و cloudflared { display-mode: "form" }
import subprocess

print('⏳ جاري تثبيت حزمة MDAAR (تشغيل قياسي بدون مكتبات GPU ثقيلة)...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)

cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ تثبيت cloudflared لنفق الربط...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

from mdaar_pipeline import __version__, FREE_GEMINI_FALLBACK_CHAIN
print(f'🎉 تم تجهيز MDAAR FreeRouterServer v{__version__}!')
print('💡 سلسلة Gemini المجانية الجاهزة:', ', '.join(FREE_GEMINI_FALLBACK_CHAIN))

In [ ]:
#@title 2. ضبط المفاتيح والمزودين وإنشاء خادم التوجيه { display-mode: "form" }
ROUTER_AUTH_TOKEN = "" #@param {type:"string"}
ROUTER_PORT = "" #@param {type:"string"}
COOLDOWN_SECONDS = "" #@param {type:"string"}

GEMINI_KEYS = "" #@param {type:"string"}
GEMINI_MODELS = "" #@param {type:"string"}

GROQ_KEYS = "" #@param {type:"string"}
GROQ_MODELS = "" #@param {type:"string"}

OPENROUTER_KEYS = "" #@param {type:"string"}
OPENROUTER_MODELS = "" #@param {type:"string"}

MISTRAL_KEYS = "" #@param {type:"string"}
MISTRAL_MODELS = "" #@param {type:"string"}

CEREBRAS_KEYS = "" #@param {type:"string"}
CEREBRAS_MODELS = "" #@param {type:"string"}

CUSTOM_NAME = "" #@param {type:"string"}
CUSTOM_BASE_URL = "" #@param {type:"string"}
CUSTOM_KEYS = "" #@param {type:"string"}
CUSTOM_MODELS = "" #@param {type:"string"}

import secrets
from mdaar_pipeline import FreeRouterServer, resolve_setting

token_val, token_src = resolve_setting(ROUTER_AUTH_TOKEN, ('MDAAR_ROUTER_TOKEN',), default='', return_source=True)
generated_token = False
if not token_val:
    token_val = secrets.token_urlsafe(24)
    token_src = 'auto-generated'
    generated_token = True

port_val = int(resolve_setting(ROUTER_PORT, ('MDAAR_ROUTER_PORT',), default='8787') or 8787)
cooldown_val = int(resolve_setting(COOLDOWN_SECONDS, ('MDAAR_ROUTER_COOLDOWN',), default='45') or 45)

gem_keys = resolve_setting(GEMINI_KEYS, ('MDAAR_ROUTER_GEMINI_KEYS', 'GEMINI_API_KEY'), default='')
gem_models = resolve_setting(GEMINI_MODELS, ('MDAAR_ROUTER_GEMINI_MODELS',), default='auto-free-chain')

groq_keys = resolve_setting(GROQ_KEYS, ('MDAAR_ROUTER_GROQ_KEYS', 'GROQ_API_KEY'), default='')
groq_models = resolve_setting(GROQ_MODELS, ('MDAAR_ROUTER_GROQ_MODELS',), default='llama-3.3-70b-versatile,llama-3.1-8b-instant')

or_keys = resolve_setting(OPENROUTER_KEYS, ('MDAAR_ROUTER_OPENROUTER_KEYS', 'OPENROUTER_API_KEY'), default='')
or_models = resolve_setting(OPENROUTER_MODELS, ('MDAAR_ROUTER_OPENROUTER_MODELS',), default='openrouter/free')

mis_keys = resolve_setting(MISTRAL_KEYS, ('MDAAR_ROUTER_MISTRAL_KEYS', 'MISTRAL_API_KEY'), default='')
mis_models = resolve_setting(MISTRAL_MODELS, ('MDAAR_ROUTER_MISTRAL_MODELS',), default='mistral-small-latest')

cer_keys = resolve_setting(CEREBRAS_KEYS, ('MDAAR_ROUTER_CEREBRAS_KEYS', 'CEREBRAS_API_KEY'), default='')
cer_models = resolve_setting(CEREBRAS_MODELS, ('MDAAR_ROUTER_CEREBRAS_MODELS',), default='llama-3.3-70b')

cust_name = resolve_setting(CUSTOM_NAME, ('MDAAR_ROUTER_CUSTOM_NAME',), default='custom')
cust_url = resolve_setting(CUSTOM_BASE_URL, ('MDAAR_ROUTER_CUSTOM_BASE_URL',), default='')
cust_keys = resolve_setting(CUSTOM_KEYS, ('MDAAR_ROUTER_CUSTOM_KEYS',), default='')
cust_models = resolve_setting(CUSTOM_MODELS, ('MDAAR_ROUTER_CUSTOM_MODELS',), default='')

if 'router_server' in globals() and router_server is not None:
    router_server.stop()

router_server = FreeRouterServer.from_providers(
    gemini_keys=gem_keys,
    gemini_models=gem_models,
    groq_keys=groq_keys,
    groq_models=groq_models,
    openrouter_keys=or_keys,
    openrouter_models=or_models,
    mistral_keys=mis_keys,
    mistral_models=mis_models,
    cerebras_keys=cer_keys,
    cerebras_models=cer_models,
    custom_name=cust_name,
    custom_base_url=cust_url,
    custom_keys=cust_keys,
    custom_models=cust_models,
    auth_token=token_val,
    host='127.0.0.1',
    port=port_val,
    cooldown_seconds=cooldown_val,
)

health_preview = router_server.get_health_payload()
print('=' * 70)
print(f'🛡️ حالة الحماية: مفعّلة ({token_src})')
if generated_token:
    print(f'🔑 تم توليد توكن تلقائي لهذه الجلسة (انسخه لدفتر MDAAR): {token_val}')
else:
    print('🔑 سيتم استخدام التوكن المحدد في MDAAR_ROUTER_TOKEN / الحقل أعلاه.')
print(f'📡 عدد المسارات النشطة: {len(health_preview["routes"])}')
for r in health_preview['routes']:
    print(f'  • {r["name"]} ({r["kind"]}) — {r["key_count"]} مفتاح | الموديلات: {", ".join(r["models"][:5])}{"..." if len(r["models"]) > 5 else ""}')
if not health_preview['routes']:
    print('⚠️ لم يتم العثور على أي مفاتيح مزودين! أضف مفتاحاً واحداً على الأقل في الحقول أو Colab Secrets.')
print('=' * 70)

In [ ]:
#@title 3. تشغيل الراوتر وفتح نفق Cloudflare وطباعة بيانات الربط { display-mode: "form" }
import json, urllib.request

public_root_url = router_server.serve_with_tunnel(timeout=30, stop_existing=True)

# Self-test local health endpoint
with urllib.request.urlopen(f'http://127.0.0.1:{router_server.bound_port}/health', timeout=5) as resp:
    health_ok = json.loads(resp.read().decode('utf-8'))

print('✅ فحص الخادم المحلي ناجح:', health_ok['status'], '| المسارات:', len(health_ok['routes']))
print('=' * 70)
if public_root_url:
    public_v1 = f'{public_root_url}/v1'
    print('🌐 رابط الراوتر العام جاهز للربط من دفتر MDAAR الرئيسي:')
    print(f'   • PROVIDER        : OpenAI-Gateway')
    print(f'   • OPENAI_BASE_URL : {public_v1}')
    print(f'   • OPENAI_MODELS   : auto')
    print('   • OPENAI_API_KEY  : التوكن الخاص بك (أو أضف MDAAR_ROUTER_URL و MDAAR_ROUTER_TOKEN في أسرار دفتر التوثيق)')
else:
    print('⚠️ تعذر استخراج رابط cloudflared، الخادم يعمل محلياً على:', router_server.base_url)
print('=' * 70)

In [ ]:
#@title 4. مراقبة الطلبات وحالة التبديل الحي (Keep-Alive & Stats) { display-mode: "form" }
MONITOR_MINUTES = 60 #@param {type:"integer"}
REFRESH_SECONDS = 30 #@param {type:"integer"}

import time, json

print(f'📡 جاري مراقبة الراوتر لمدة {MONITOR_MINUTES} دقيقة (اضغط Stop لإيقاف المراقبة فقط مع إبقاء الخادم يعمل)...')
end_time = time.time() + max(1, int(MONITOR_MINUTES)) * 60
try:
    while time.time() < end_time:
        stats = router_server.get_health_payload()['stats']
        print(
            f'[{time.strftime("%H:%M:%S")}] إجمالي={stats["total_requests"]} | '
            f'نجاح={stats["success_requests"]} | فشل={stats["failed_requests"]} | '
            f'تهدئة نشطة={stats["active_cooldowns"]} | حسب المزود={json.dumps(stats["successes_by_route"], ensure_ascii=False)}'
        )
        time.sleep(max(5, int(REFRESH_SECONDS)))
except KeyboardInterrupt:
    print('⏹️ تم إيقاف طباعة المراقبة (الخادم والنفق ما زالا يعملان بالخلفية).')

In [ ]:
#@title 5. إيقاف الراوتر والنفق بشكل نظيف (اختياري) { display-mode: "form" }
if 'router_server' in globals() and router_server is not None:
    router_server.stop()
    print('🛑 تم إيقاف خادم الراوتر ونفق Cloudflare بنجاح.')